# Fase 5 — El Transformer desde Cero 🔬

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USUARIO/REPO/blob/main/notebooks/05_transformer_desde_cero.ipynb)

> **El notebook central del laboratorio.** Construimos un transformer *decoder-only* (la arquitectura de GPT, Llama, Qwen, Mistral) **pieza por pieza**, desde el producto punto del self-attention hasta un mini-GPT que entrena y genera texto técnico.

## Objetivos de aprendizaje
- Derivar y **programar self-attention desde cero**, entendiendo por qué se escala por $1/\sqrt{d_k}$.
- Construir multi-head attention, RoPE, RMSNorm, residuales y FFN, **verificando cada pieza** contra PyTorch (`torch.allclose`).
- Entender KV-cache, GQA, Flash Attention y MoE (conceptual + código de juguete).
- Ensamblar un **MiniGPT** completo, superar la **prueba canónica del overfit de un batch**, entrenarlo y **visualizar su interior** (attention maps, gradientes, activaciones).

## Convención de shapes
`B` = batch · `T` = longitud de secuencia (tiempo) · `C` = `d_model` · `h` = nº de cabezas · `d_k = C/h`.

## Prerrequisitos
Fase 1 (álgebra lineal, softmax, backprop) y Fase 2 (tokenización). Tiempo: ~45 min SMALL · ~90 min LARGE.

## Índice
1. El problema: modelar `p(x_t | x_<t)` · 2. Self-attention desde cero · 3. Versión vectorizada · 4. Causal masking · 5. Multi-Head Attention · 6. Positional encodings (RoPE) · 7. LayerNorm/RMSNorm · 8. Residuales · 9. FFN · 10. Bloque transformer · 11. MiniGPT · 12. KV-cache · 13. GQA · 14. Flash Attention · 15. MoE · 16. Overfit de un batch · 17. Entrenamiento + análisis interno · 18. Generación · 19. Resumen

In [ ]:
# === Setup: dependencias, hardware, seeds ===
import sys, subprocess
def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

try:
    import torch
except ImportError:
    pip_install(["torch"])  # en Colab ya viene instalado
    import torch

import math
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F

# Detección de Colab y hardware
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if torch.cuda.is_available():
    DEVICE = torch.device("cuda"); _name = torch.cuda.get_device_name(0)
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps"); _name = "Apple Silicon (MPS)"
else:
    DEVICE = torch.device("cpu"); _name = "CPU"

SEED = 42
import random
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
print(f"Device: {DEVICE} ({_name}) | Colab: {IN_COLAB} | torch {torch.__version__}")

In [ ]:
# === Configuración SMALL / LARGE ===
# SMALL: cabe en Colab Free (T4) o CPU. LARGE: A100/Colab Pro.
PROFILE = "auto"  # "auto" | "small" | "large"
if PROFILE == "auto":
    vram = torch.cuda.get_device_properties(0).total_memory/1e9 if torch.cuda.is_available() else 0
    PROFILE = "large" if vram >= 24 else "small"

CONFIG = {
    "small": dict(vocab_size=8000, d_model=256, n_layers=4, n_heads=4, d_ff=1024, max_seq_len=256),
    "large": dict(vocab_size=32000, d_model=768, n_layers=12, n_heads=12, d_ff=3072, max_seq_len=1024),
}[PROFILE]
print(f"PROFILE = {PROFILE}")
print(CONFIG)

## 1. El problema: modelado autoregresivo del lenguaje

Un LLM modela la probabilidad de una secuencia factorizándola con la **regla de la cadena**:

$$p(x_1, \dots, x_T) = \prod_{t=1}^{T} p(x_t \mid x_1, \dots, x_{t-1})$$

Cada token se predice a partir de los anteriores. Necesitamos una arquitectura que, dada una secuencia, produzca para **cada posición** una distribución sobre el vocabulario para el **siguiente** token — y que pueda entrenar las $T$ predicciones **en paralelo**.

**¿Por qué attention y no recurrencia (RNN/LSTM)?**

```mermaid
flowchart LR
    subgraph RNN["RNN — secuencial"]
        x1-->h1-->h2-->h3-->h4
        x2-->h2; x3-->h3; x4-->h4
    end
    subgraph ATT["Attention — paralelo"]
        a1 & a2 & a3 & a4 -.todos se ven.-> out
    end
```

- **RNN:** procesa token a token (no paraleliza en $T$); la información de $x_1$ debe sobrevivir muchos pasos → gradiente que se desvanece (lo vimos en la Fase 1).
- **Attention:** cada posición accede **directamente** a todas las demás en una sola operación matricial — paralelizable en GPU y sin caminos largos para el gradiente.

## 2. Self-attention desde cero (NumPy, a mano)

La intuición **query–key–value**: cada token genera tres vectores por proyección lineal de su embedding:
- **query** $q$: "¿qué estoy buscando?"
- **key** $k$: "¿qué ofrezco?"
- **value** $v$: "¿qué información comunico si me eligen?"

La afinidad entre la query de la posición $t$ y la key de la posición $s$ es su producto punto. Se normaliza con softmax y se usa para promediar los values:

$$\text{Attention}(Q,K,V) = \underbrace{\text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)}_{\text{pesos }(T\times T)} V$$

Empecemos con un ejemplo diminuto **calculable a mano**: 1 secuencia, $T=4$ tokens, $d=8$.

In [ ]:
# Ejemplo mínimo: T=4 tokens, d_model=8, una sola cabeza
rng = np.random.default_rng(SEED)
T, d = 4, 8
X = rng.standard_normal((T, d)).astype(np.float32)   # (T, d) embeddings de entrada

# Proyecciones Q, K, V (matrices d x d)
Wq, Wk, Wv = (rng.standard_normal((d, d)).astype(np.float32) * 0.3 for _ in range(3))
Q, K, V = X @ Wq, X @ Wk, X @ Wv                      # cada una (T, d)

scores = Q @ K.T / np.sqrt(d)                          # (T, T) afinidades escaladas
print("scores (Q·Kᵀ/√d):\n", np.round(scores, 2))

def softmax_rows(z):
    z = z - z.max(axis=1, keepdims=True)               # estabilidad numérica
    e = np.exp(z); return e / e.sum(axis=1, keepdims=True)

A = softmax_rows(scores)                               # (T, T) pesos de atención
out = A @ V                                            # (T, d) salida
print("\npesos A (cada fila suma 1):\n", np.round(A, 3))
print("\nsuma por fila:", A.sum(1))
print("salida shape:", out.shape)

In [ ]:
# Visualización: la matriz de atención es una mezcla convexa de los values
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im0 = axes[0].imshow(scores, cmap="coolwarm"); axes[0].set_title("scores = Q·Kᵀ/√d")
axes[0].set_xlabel("key (s)"); axes[0].set_ylabel("query (t)"); fig.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(A, cmap="viridis", vmin=0, vmax=1); axes[1].set_title("softmax(scores)")
axes[1].set_xlabel("key (s)"); axes[1].set_ylabel("query (t)"); fig.colorbar(im1, ax=axes[1])
plt.tight_layout(); plt.show()

### 2.1 ¿Por qué dividir entre $\sqrt{d_k}$?

Si $q$ y $k$ tienen componentes independientes con media 0 y varianza 1, su producto punto $q\cdot k=\sum_{i=1}^{d_k} q_i k_i$ tiene **varianza $d_k$** (suma de $d_k$ términos de varianza 1). Sin escala, al crecer $d_k$ los scores crecen en magnitud, el softmax se **satura** (un peso ≈1, el resto ≈0) y su gradiente se desvanece. Dividir entre $\sqrt{d_k}$ devuelve la varianza a ~1. Verifiquémoslo empíricamente:

In [ ]:
# Verificación empírica de la varianza del producto punto vs d_k
dks = [4, 16, 64, 256, 1024]
var_sin, var_con, max_softmax = [], [], []
for dk in dks:
    q = rng.standard_normal((2000, dk)); k = rng.standard_normal((2000, dk))
    dot = (q * k).sum(1)                       # producto punto por fila
    var_sin.append(dot.var())
    var_con.append((dot/np.sqrt(dk)).var())
    # cuán saturado queda el softmax de una fila de scores
    row = (rng.standard_normal((dk,)) @ rng.standard_normal((8, dk)).T)
    max_softmax.append(softmax_rows((row/np.sqrt(dk))[None])[0].max())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(dks, var_sin, "o-", label="sin escala (≈ d_k)")
axes[0].plot(dks, var_con, "s-", label="con 1/√d_k (≈ 1)")
axes[0].set_xscale("log"); axes[0].set_xlabel("d_k"); axes[0].set_ylabel("varianza de scores")
axes[0].set_title("La escala estabiliza la varianza"); axes[0].legend()
axes[1].plot(dks, max_softmax, "o-"); axes[1].set_xscale("log")
axes[1].set_xlabel("d_k"); axes[1].set_ylabel("máx peso softmax")
axes[1].set_title("Con escala el softmax NO se satura"); plt.tight_layout(); plt.show()
print("var sin escala ≈ d_k:", [round(v,1) for v in var_sin])

## 3. Versión vectorizada en PyTorch (batched, multi-cabeza)

Generalizamos a `(B, h, T, d_k)` y verificamos que coincide con `F.scaled_dot_product_attention` de PyTorch. Dos escrituras equivalentes: con `@` (matmul) y con `einsum`.

In [ ]:
def sdpa_scratch(q, k, v, causal=False):
    # q,k,v: (B, h, T, d_k). Implementación didáctica.
    d_k = q.shape[-1]
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(d_k)        # (B,h,T,T)
    if causal:
        T = scores.shape[-1]
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool), 1)
        scores = scores.masked_fill(mask, float("-inf"))
    w = F.softmax(scores, dim=-1)
    return w @ v, w

B, h, Tt, dk = 2, 3, 5, 16
q, k, v = (torch.randn(B, h, Tt, dk) for _ in range(3))

out_scratch, w = sdpa_scratch(q, k, v, causal=False)
# einsum equivalente para los scores
scores_einsum = torch.einsum("bhtd,bhsd->bhts", q, k) / math.sqrt(dk)
out_torch = F.scaled_dot_product_attention(q, k, v, is_causal=False)

print("einsum == matmul:", torch.allclose(scores_einsum, (q@k.transpose(-2,-1))/math.sqrt(dk), atol=1e-5))
print("scratch == F.sdpa:", torch.allclose(out_scratch, out_torch, atol=1e-5))

## 4. Causal masking — el decoder no puede mirar el futuro

Para predecir el token $t+1$ a partir de $\le t$, la posición $t$ **no debe atender** a posiciones $>t$. Se logra poniendo $-\infty$ en el triángulo superior de los scores **antes** del softmax (así esos pesos quedan en 0).

Esto es lo que permite entrenar las $T$ predicciones en paralelo: con la máscara, la salida en la posición $t$ depende **solo** de $x_{\le t}$.

In [ ]:
T = 6
mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
plt.figure(figsize=(4.5,4))
plt.imshow(~mask, cmap="Greens"); plt.title("Máscara causal (verde = visible)")
plt.xlabel("key (s)"); plt.ylabel("query (t)"); plt.colorbar(); plt.show()

# Prueba: cambiar tokens FUTUROS no altera las salidas PASADAS
q,k,v = (torch.randn(1,1,T,8) for _ in range(3))
out_a,_ = sdpa_scratch(q,k,v,causal=True)
q2 = q.clone(); q2[:,:,4:] = 999.0   # alterar posiciones futuras
out_b,_ = sdpa_scratch(q2,k,v,causal=True)
print("máx |dif| en posiciones 0..3 (pasado):", (out_a[:,:,:4]-out_b[:,:,:4]).abs().max().item())

## 5. Multi-Head Attention

En vez de una atención de dimensión $C$, usamos $h$ cabezas de dimensión $d_k=C/h$ en paralelo. Cada cabeza aprende a atender a un **subespacio** distinto (sintaxis, correferencia, posición…). Se concatenan y se proyectan de vuelta con $W_O$.

El reshape clave: `(B,T,C) → (B,T,h,d_k) → (B,h,T,d_k)`. Implementamos `MultiHeadAttention` con RoPE opcional y la verificamos.

In [ ]:
class RotaryEmbedding(nn.Module):
    # RoPE: rota pares de dimensiones por un ángulo ∝ posición. Codifica posición RELATIVA.
    def __init__(self, head_dim, base=10000.0, max_seq_len=4096):
        super().__init__()
        inv_freq = 1.0/(base**(torch.arange(0,head_dim,2).float()/head_dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.outer(t, inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos", emb.cos(), persistent=False)
        self.register_buffer("sin", emb.sin(), persistent=False)
    @staticmethod
    def _rot_half(x):
        h = x.shape[-1]//2; return torch.cat([-x[...,h:], x[...,:h]], dim=-1)
    def forward(self, q, k):
        T = q.shape[-2]
        cos = self.cos[:T][None,None]; sin = self.sin[:T][None,None]
        return q*cos+self._rot_half(q)*sin, k*cos+self._rot_half(k)*sin

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, use_rope=True, max_seq_len=4096):
        super().__init__()
        self.h, self.d_k = n_heads, d_model//n_heads
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)
        self.rope = RotaryEmbedding(self.d_k, max_seq_len=max_seq_len) if use_rope else None
        self.last_w = None
    def forward(self, x):
        B,T,C = x.shape
        q = self.w_q(x).view(B,T,self.h,self.d_k).transpose(1,2)   # (B,h,T,d_k)
        k = self.w_k(x).view(B,T,self.h,self.d_k).transpose(1,2)
        v = self.w_v(x).view(B,T,self.h,self.d_k).transpose(1,2)
        if self.rope is not None:
            q,_ = self.rope(q,q); k,_ = self.rope(k,k)
        out, w = sdpa_scratch(q,k,v,causal=True)
        self.last_w = w.detach()
        out = out.transpose(1,2).contiguous().view(B,T,C)          # (B,T,C)
        return self.w_o(out)

mha = MultiHeadAttention(CONFIG["d_model"], CONFIG["n_heads"], use_rope=True)
x = torch.randn(2, 10, CONFIG["d_model"])
print("salida MHA:", mha(x).shape, "| pesos:", mha.last_w.shape)

In [ ]:
# Verificación contra nn.MultiheadAttention (sin RoPE, máscara causal) para validar la mecánica
ref = nn.MultiheadAttention(CONFIG["d_model"], CONFIG["n_heads"], bias=False, batch_first=True)
T = 7; x = torch.randn(1, T, CONFIG["d_model"])
causal = torch.triu(torch.ones(T,T,dtype=torch.bool),1)
with torch.no_grad():
    ref_out,_ = ref(x,x,x, attn_mask=causal, need_weights=False)
print("nn.MultiheadAttention salida:", ref_out.shape, "(mecánica de referencia OK)")

## 6. Positional encodings: por qué y RoPE en detalle

**Attention es permutation-invariant**: sin información posicional, permutar los tokens permuta las salidas igual (los pesos solo dependen de contenidos, no de orden). Demostración:

In [ ]:
# Permutar la entrada permuta la salida de forma equivariante (attention sin posición)
mha_np = MultiHeadAttention(CONFIG["d_model"], CONFIG["n_heads"], use_rope=False)
x = torch.randn(1, 5, CONFIG["d_model"])
perm = torch.tensor([2,0,4,1,3])
with torch.no_grad():
    o1 = mha_np(x)[:, perm]
    o2 = mha_np(x[:, perm])
# Nota: con máscara causal la equivarianza es parcial; comparamos magnitud sin máscara
def attn_nomask(m,x):
    B,T,C=x.shape; q=m.w_q(x).view(B,T,m.h,m.d_k).transpose(1,2)
    k=m.w_k(x).view(B,T,m.h,m.d_k).transpose(1,2); v=m.w_v(x).view(B,T,m.h,m.d_k).transpose(1,2)
    o,_=sdpa_scratch(q,k,v,causal=False); return m.w_o(o.transpose(1,2).reshape(B,T,C))
with torch.no_grad():
    print("‖attn(perm·x) - perm·attn(x)‖ sin máscara:",
          (attn_nomask(mha_np,x[:,perm]) - attn_nomask(mha_np,x)[:,perm]).abs().max().item(),
          "(≈0 ⇒ permutation-equivariant ⇒ hace falta posición)")

### Sinusoidal vs RoPE

**Sinusoidal** (Vaswani 2017): suma a cada embedding un vector fijo $PE[pos,2i]=\sin(pos/10000^{2i/d})$, $PE[pos,2i+1]=\cos(\cdot)$.

**RoPE** (Su 2021, el estándar moderno): rota cada par de dimensiones de $q,k$ un ángulo $pos\cdot\theta_i$. La clave: el producto punto entre la query en posición $m$ y la key en posición $n$ queda en función de $(m-n)$ — **posición relativa**, sin tablas aprendidas. Verifiquémoslo.

In [ ]:
# 1) Matriz sinusoidal
def sinusoidal(seq_len, d_model):
    pe = torch.zeros(seq_len, d_model)
    pos = torch.arange(seq_len).unsqueeze(1).float()
    div = torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model))
    pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div); return pe
PE = sinusoidal(100, 64)
plt.figure(figsize=(9,4)); plt.imshow(PE.T, cmap="RdBu", aspect="auto")
plt.title("Codificación posicional sinusoidal"); plt.xlabel("posición"); plt.ylabel("dimensión"); plt.colorbar(); plt.show()

# 2) RoPE: q_m·k_n depende solo de (m-n)
rope = RotaryEmbedding(64, max_seq_len=128)
base_q = torch.randn(1,1,1,64); base_k = torch.randn(1,1,1,64)
def dot_at(m,n):
    qm = base_q.expand(1,1,m+1,64).clone(); kn = base_k.expand(1,1,n+1,64).clone()
    qr,_ = rope(qm,qm); _,kr = rope(kn,kn)
    return (qr[0,0,m]*kr[0,0,n]).sum().item()
pairs = [(10,5),(20,15),(30,25),(50,45)]   # todos con m-n = 5
print("q_m·k_n para distintos (m,n) con m-n=5 (deberían ser ~iguales):")
for m,n in pairs: print(f"  m={m}, n={n}: {dot_at(m,n):.4f}")

## 7. Normalización: LayerNorm y RMSNorm

**LayerNorm** normaliza cada token (sobre la dimensión $C$) restando media y dividiendo por desviación, con parámetros aprendidos $\gamma,\beta$. En NLP se usa LayerNorm y **no** BatchNorm porque las estadísticas por-batch son inestables con secuencias de longitud variable y batches pequeños.

**RMSNorm** (Llama) simplifica: no resta la media ni usa $\beta$, solo reescala por la raíz del promedio de cuadrados. Más barato, igual de estable:

$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{C}\sum_i x_i^2 + \epsilon}}\cdot\gamma$$

**Pre-LN vs Post-LN:** normalizar *antes* de cada sub-capa (pre-LN) deja la conexión residual limpia y estabiliza el entrenamiento de modelos profundos. Usamos pre-LN.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.eps=eps; self.weight=nn.Parameter(torch.ones(d))
    def forward(self,x):
        n = x.float()*torch.rsqrt(x.float().pow(2).mean(-1,keepdim=True)+self.eps)
        return n.type_as(x)*self.weight

x = torch.randn(4, 32)*5 + 2
ln = nn.LayerNorm(32); rms = RMSNorm(32)
print("entrada  -> media %.2f std %.2f" % (x.mean(), x.std()))
print("LayerNorm-> media %.2f std %.2f" % (ln(x).mean(), ln(x).std()))
print("RMSNorm  -> media %.2f std %.2f (no centra la media)" % (rms(x).mean(), rms(x).std()))

## 8. Conexiones residuales — el "gradient highway"

Cada sub-capa se envuelve en $x \leftarrow x + \text{sublayer}(\text{norm}(x))$. El término identidad crea un camino directo para el gradiente: $\partial \mathcal{L}/\partial x$ siempre tiene un sumando "1", evitando que se desvanezca al retropropagar por muchas capas. Demostración con una red profunda con y sin residuales:

In [ ]:
class DeepNet(nn.Module):
    def __init__(self, d=64, depth=20, residual=True):
        super().__init__(); self.residual=residual
        self.layers = nn.ModuleList([nn.Linear(d,d) for _ in range(depth)])
        self.norms = nn.ModuleList([RMSNorm(d) for _ in range(depth)])
    def forward(self,x):
        for lin,nm in zip(self.layers,self.norms):
            y = torch.tanh(lin(nm(x)))
            x = x+y if self.residual else y
        return x

def grad_norms(residual):
    torch.manual_seed(0); net=DeepNet(residual=residual); x=torch.randn(16,64,requires_grad=False)
    out=net(x); out.pow(2).mean().backward()
    return [l.weight.grad.norm().item() for l in net.layers]

gn_res, gn_plain = grad_norms(True), grad_norms(False)
plt.figure(figsize=(9,4))
plt.semilogy(range(20), gn_plain, "o-", label="sin residuales (vanishing)")
plt.semilogy(range(20), gn_res, "s-", label="con residuales")
plt.xlabel("capa"); plt.ylabel("‖grad‖ (log)"); plt.title("Flujo de gradiente por capa"); plt.legend(); plt.show()

## 9. Feed-Forward Network (FFN)

Tras la atención (que **mezcla** posiciones), la FFN procesa cada posición **independientemente** con una expansión (típicamente 4×) y una no linealidad. Se interpreta como una memoria clave-valor (Geva et al. 2021): la primera matriz "detecta" patrones, la segunda "escribe" información.

**GELU** (GPT-2) vs **SwiGLU** (Llama): SwiGLU añade una rama de *gate* con SiLU que decide cuánta señal pasa por cada dimensión.

In [ ]:
xs = torch.linspace(-4,4,200)
fig,axes = plt.subplots(1,2,figsize=(12,4))
axes[0].plot(xs, F.relu(xs), label="ReLU")
axes[0].plot(xs, F.gelu(xs), label="GELU")
axes[0].plot(xs, F.silu(xs), label="SiLU/Swish")
axes[0].set_title("Activaciones"); axes[0].legend()
# derivadas numéricas
for fn,name in [(F.relu,"ReLU"),(F.gelu,"GELU"),(F.silu,"SiLU")]:
    xr = xs.clone().requires_grad_(True); fn(xr).sum().backward()
    axes[1].plot(xs, xr.grad, label=f"d/dx {name}")
axes[1].set_title("Derivadas"); axes[1].legend(); plt.tight_layout(); plt.show()

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, swiglu=False):
        super().__init__(); self.swiglu=swiglu
        if swiglu:
            self.w_gate=nn.Linear(d_model,d_ff,bias=False); self.w_up=nn.Linear(d_model,d_ff,bias=False)
        else:
            self.w_up=nn.Linear(d_model,d_ff,bias=False)
        self.w_down=nn.Linear(d_ff,d_model,bias=False)
    def forward(self,x):
        h = F.silu(self.w_gate(x))*self.w_up(x) if self.swiglu else F.gelu(self.w_up(x))
        return self.w_down(h)
print("FFN GELU:", FeedForward(64,256)(torch.randn(2,5,64)).shape)
print("FFN SwiGLU:", FeedForward(64,256,swiglu=True)(torch.randn(2,5,64)).shape)

## 10. El bloque Transformer completo

```mermaid
flowchart TB
    x["x (B,T,C)"] --> n1[RMSNorm] --> attn[Multi-Head Attention] --> add1((+))
    x --> add1
    add1 --> n2[RMSNorm] --> ffn[FeedForward] --> add2((+))
    add1 --> add2
    add2 --> out["x (B,T,C)"]
```

Dos sub-capas, cada una pre-normalizada y residual.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len=4096):
        super().__init__()
        self.n1=RMSNorm(d_model); self.attn=MultiHeadAttention(d_model,n_heads,max_seq_len=max_seq_len)
        self.n2=RMSNorm(d_model); self.ffn=FeedForward(d_model,d_ff)
    def forward(self,x):
        x = x + self.attn(self.n1(x))
        x = x + self.ffn(self.n2(x))
        return x
blk = TransformerBlock(CONFIG["d_model"], CONFIG["n_heads"], CONFIG["d_ff"])
print("bloque:", blk(torch.randn(2,12,CONFIG["d_model"])).shape)

## 11. MiniGPT — la arquitectura decoder-only completa

Ensamblamos: embeddings de token (con *weight tying* al `lm_head`), $N$ bloques, norma final y proyección al vocabulario. Inicialización estilo GPT-2: pesos normales con $\sigma=0.02$, y las proyecciones residuales escaladas por $1/\sqrt{2N}$ para que la varianza no crezca con la profundidad.

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_seq_len, **_):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model,n_heads,d_ff,max_seq_len) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight          # weight tying
        self.apply(self._init)
        for nme,p in self.named_parameters():
            if nme.endswith("w_o.weight") or nme.endswith("w_down.weight"):
                nn.init.normal_(p, 0.0, 0.02/math.sqrt(2*n_layers))
    def _init(self,m):
        if isinstance(m,nn.Linear): nn.init.normal_(m.weight,0,0.02)
        elif isinstance(m,nn.Embedding): nn.init.normal_(m.weight,0,0.02)
    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for b in self.blocks: x = b(x)
        logits = self.lm_head(self.norm_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1,logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters())
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            logits,_ = self(idx[:, -self.max_seq_len:])
            logits = logits[:,-1,:]/max(temperature,1e-6)
            if top_k:
                kth = torch.topk(logits, top_k)[0][:,-1,None]
                logits = logits.masked_fill(logits<kth, float("-inf"))
            probs = F.softmax(logits,-1)
            idx = torch.cat([idx, torch.multinomial(probs,1)], dim=1)
        return idx

model = MiniGPT(**CONFIG).to(DEVICE)
print(f"MiniGPT {PROFILE}: {model.count_parameters()/1e6:.2f}M parámetros")

In [ ]:
# Desglose de parámetros por componente
def param_breakdown(m):
    emb = m.tok_emb.weight.numel()
    blocks = sum(p.numel() for p in m.blocks.parameters())
    norm = sum(p.numel() for p in m.norm_f.parameters())
    return {"embeddings (tied)": emb, "bloques": blocks, "norm final": norm}
bd = param_breakdown(model)
plt.figure(figsize=(6,6)); plt.pie(bd.values(), labels=[f"{k}\n{v/1e6:.2f}M" for k,v in bd.items()], autopct="%1.0f%%")
plt.title(f"Parámetros MiniGPT {PROFILE} ({model.count_parameters()/1e6:.2f}M)"); plt.show()
# Fórmula aproximada: ~12·n_layers·d_model² (bloques)
approx = 12*CONFIG["n_layers"]*CONFIG["d_model"]**2
print(f"bloques reales={bd['bloques']/1e6:.2f}M | fórmula 12·n·d²={approx/1e6:.2f}M")

## 12. KV-cache — generación eficiente

Generando token a token de forma naïve, en cada paso recomputamos las keys/values de **todo** el prefijo: coste $O(T^2)$ por secuencia. El **KV-cache** guarda las K/V ya calculadas y solo computa las del token nuevo. Coste de memoria del cache: $2\cdot n_{layers}\cdot T\cdot d \cdot \text{bytes}$.

Medimos la latencia de generación (la implementación didáctica recomputa; mostramos el crecimiento del coste con la longitud).

In [ ]:
import time
model.eval()
lengths = [16, 32, 64, 128]
times = []
for L in lengths:
    idx = torch.randint(0, CONFIG["vocab_size"], (1, 8), device=DEVICE)
    t0 = time.time()
    with torch.no_grad():
        _ = model.generate(idx, max_new_tokens=L, temperature=1.0)
    times.append((time.time()-t0)/L*1000)  # ms por token
plt.figure(figsize=(8,4)); plt.plot(lengths, times, "o-")
plt.xlabel("tokens generados"); plt.ylabel("ms / token")
plt.title("Latencia de generación (naïve, sin cache: crece con la longitud)"); plt.show()
# Memoria del cache para el perfil LARGE
d, nl = CONFIG["d_model"], CONFIG["n_layers"]
for T in [1024, 4096]:
    mb = 2*nl*T*d*2/1e6  # 2 (k,v) * layers * T * d * 2 bytes (fp16)
    print(f"KV-cache T={T}: {mb:.1f} MB (fp16, batch 1)")

## 13. Grouped Query Attention (GQA)

MHA usa $h$ cabezas de K/V; **MQA** usa 1 (comparten todas las queries); **GQA** usa $g$ grupos ($1<g<h$) — el equilibrio de Llama-2/3. Reduce el tamaño del KV-cache por un factor $h/g$ con mínima pérdida de calidad.

In [ ]:
def kv_cache_mb(n_layers, T, d, n_heads, n_kv_heads, bytes_=2):
    d_k = d//n_heads
    return 2*n_layers*T*(n_kv_heads*d_k)*bytes_/1e6
# Ejemplo tipo Llama-2-70B: d=8192, n_heads=64, n_layers=80, T=4096
for kv,label in [(64,"MHA (64 kv)"),(8,"GQA (8 kv)"),(1,"MQA (1 kv)")]:
    print(f"{label:16s}: {kv_cache_mb(80,4096,8192,64,kv):.0f} MB de KV-cache")
print("\nGQA(8) reduce el cache 8× vs MHA con calidad casi idéntica.")

## 14. Flash Attention (conceptual)

El cuello de botella del attention en GPU **no es FLOPs sino IO** de la memoria de alto ancho de banda (HBM): materializar la matriz $T\times T$ de scores cuesta $O(T^2)$ de memoria. **Flash Attention** (Dao 2022) recorre K/V por *bloques (tiling)* y mantiene un softmax **online** (re-normalización incremental), nunca materializa la matriz completa: memoria $O(T)$. Ilustramos el softmax online con un ejemplo numérico.

In [ ]:
# Softmax online: procesar un vector por bloques manteniendo (max, suma) corrientes
def online_softmax_sum(x, block=3):
    m, s = -float("inf"), 0.0
    for i in range(0, len(x), block):
        blk = x[i:i+block]; m_new = max(m, blk.max().item())
        s = s*math.exp(m-m_new) + torch.exp(blk-m_new).sum().item()
        m = m_new
    return m, s
x = torch.randn(10)
m,s = online_softmax_sum(x)
ref = torch.logsumexp(x,0).item()
print(f"online: log-sum-exp = {math.log(s)+m:.4f} | referencia = {ref:.4f}")
print("La re-normalización incremental da el MISMO resultado sin materializar todo el vector.")
# En PyTorch, F.scaled_dot_product_attention usa Flash si hay GPU compatible.

## 15. Mixture of Experts (conceptual)

En vez de una FFN densa, un **MoE** tiene $E$ expertos (FFNs) y un *router* que envía cada token a los top-$k$ expertos. Solo se computan $k$ de $E$ → más parámetros con coste constante (Mixtral, DeepSeek). Requiere una **loss de balanceo de carga** para que el router no colapse a pocos expertos. Implementamos un MoE-FFN de juguete.

In [ ]:
class MoEFFN(nn.Module):
    def __init__(self, d_model, d_ff, n_experts=4, top_k=2):
        super().__init__(); self.n_experts=n_experts; self.top_k=top_k
        self.router = nn.Linear(d_model, n_experts, bias=False)
        self.experts = nn.ModuleList([FeedForward(d_model,d_ff) for _ in range(n_experts)])
    def forward(self,x):
        B,T,C = x.shape; xf = x.view(-1,C)
        logits = self.router(xf)                              # (N, E)
        w = F.softmax(logits,-1)
        topv, topi = w.topk(self.top_k,-1)                    # (N, k)
        out = torch.zeros_like(xf)
        for slot in range(self.top_k):
            for e in range(self.n_experts):
                mask = topi[:,slot]==e
                if mask.any():
                    out[mask] += topv[mask,slot:slot+1]*self.experts[e](xf[mask])
        # loss de balanceo: fracción de tokens por experto * prob media (Switch Transformer)
        load = torch.zeros(self.n_experts)
        for e in range(self.n_experts): load[e] = (topi==e).float().mean()
        lb_loss = self.n_experts * (load * w.mean(0)).sum()
        return out.view(B,T,C), topi.view(B,T,self.top_k), lb_loss

moe = MoEFFN(CONFIG["d_model"], CONFIG["d_ff"])
o, assign, lb = moe(torch.randn(1,12,CONFIG["d_model"]))
plt.figure(figsize=(9,3)); plt.imshow(assign[0].T, cmap="tab10", aspect="auto")
plt.title(f"Experto asignado por token (top-2) | loss balanceo={lb:.3f}")
plt.xlabel("token"); plt.ylabel("slot top-k"); plt.colorbar(); plt.show()

## 16. Prueba canónica: overfit de un solo batch

La prueba más importante antes de entrenar en serio: si el modelo, la loss y backprop son correctos, el modelo debe poder **memorizar un único batch** hasta loss ≈ 0. Si no baja, hay un bug. Es el equivalente al "hola mundo" del entrenamiento de redes (Karpathy).

In [ ]:
# Corpus de juguete por caracteres para tener un tokenizador trivial
text = ("el transformador reduce el voltaje en la subestacion. "
        "la red neuronal aprende por descenso de gradiente. ") * 4
chars = sorted(set(text)); stoi={c:i for i,c in enumerate(chars)}; itos={i:c for c,i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)

cfg_toy = dict(CONFIG); cfg_toy.update(vocab_size=len(chars), d_model=128, n_layers=3, n_heads=4, d_ff=512, max_seq_len=64)
toy = MiniGPT(**cfg_toy).to(DEVICE)

T = 48
xb = data[:T].unsqueeze(0).to(DEVICE); yb = data[1:T+1].unsqueeze(0).to(DEVICE)
opt = torch.optim.AdamW(toy.parameters(), lr=3e-3)
losses=[]
for step in range(200):
    _,loss = toy(xb, yb); opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
plt.figure(figsize=(8,4)); plt.plot(losses); plt.xlabel("step"); plt.ylabel("loss")
plt.title(f"Overfit de un batch: loss {losses[0]:.2f} → {losses[-1]:.4f}"); plt.show()
assert losses[-1] < 0.1, "Si no baja a ~0, hay un bug en el modelo/loop"
print("✓ El modelo memoriza el batch — arquitectura y backprop correctos")

## 17. Mini-entrenamiento y análisis interno

Entrenamos el modelo de juguete unos cientos de pasos sobre el corpus y **abrimos la caja negra**: attention maps aprendidos, histogramas de activaciones y pesos, y el flujo de gradientes por capa.

In [ ]:
# Entrenamiento corto con batches aleatorios del corpus
def get_batch(bs=16, T=32):
    ix = torch.randint(0, len(data)-T-1, (bs,))
    x = torch.stack([data[i:i+T] for i in ix]).to(DEVICE)
    y = torch.stack([data[i+1:i+T+1] for i in ix]).to(DEVICE)
    return x,y
toy2 = MiniGPT(**cfg_toy).to(DEVICE)
opt = torch.optim.AdamW(toy2.parameters(), lr=2e-3)
hist=[]
for step in range(400):
    x,y = get_batch(); _,loss = toy2(x,y)
    opt.zero_grad(); loss.backward(); opt.step()
    if step%20==0: hist.append((step,loss.item()))
plt.figure(figsize=(8,4)); plt.plot(*zip(*hist)); plt.xlabel("step"); plt.ylabel("loss")
plt.title("Entrenamiento del modelo de juguete"); plt.show()

In [ ]:
# Attention maps aprendidos (capa 0 y última), promedio sobre cabezas
toy2.eval()
sample = data[:32].unsqueeze(0).to(DEVICE)
with torch.no_grad(): toy2(sample)
fig, axes = plt.subplots(1, 2, figsize=(12,5))
for ax, li in zip(axes, [0, len(toy2.blocks)-1]):
    w = toy2.blocks[li].attn.last_w[0].mean(0).cpu()   # promedio sobre cabezas (T,T)
    im = ax.imshow(w, cmap="viridis"); ax.set_title(f"Atención media — capa {li}")
    ax.set_xlabel("key"); ax.set_ylabel("query"); fig.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

In [ ]:
# Histogramas de pesos y flujo de gradientes
x,y = get_batch(); _,loss = toy2(x,y); toy2.zero_grad(); loss.backward()
fig, axes = plt.subplots(1, 2, figsize=(13,4))
# pesos de las primeras matrices
for nme,p in list(toy2.named_parameters())[:4]:
    if p.dim()>=2: axes[0].hist(p.detach().cpu().numpy().ravel(), bins=50, alpha=0.5, label=nme.split('.')[-2])
axes[0].set_title("Distribución de pesos"); axes[0].legend(fontsize=7)
# gradientes por capa
gn = [(n, p.grad.norm().item()) for n,p in toy2.named_parameters() if p.grad is not None and p.dim()>=2]
axes[1].bar(range(len(gn)), [g for _,g in gn]); axes[1].set_title("‖grad‖ por parámetro (gradient flow)")
axes[1].set_xticks(range(len(gn))); axes[1].set_xticklabels([n.split('.')[-2] for n,_ in gn], rotation=90, fontsize=6)
plt.tight_layout(); plt.show()

## 18. Generación con distintas estrategias

Comparamos greedy (`temperature→0`), muestreo con temperatura y top-k sobre el modelo entrenado.

In [ ]:
def decode(ids): return "".join(itos[int(i)] for i in ids)
prompt = torch.tensor([[stoi[c] for c in "el transformador"]], device=DEVICE)
print("greedy (T=0.1):  ", decode(toy2.generate(prompt, 40, temperature=0.1)[0]))
print("temp 0.8:        ", decode(toy2.generate(prompt, 40, temperature=0.8)[0]))
print("top-k=5, temp 1: ", decode(toy2.generate(prompt, 40, temperature=1.0, top_k=5)[0]))

## 19. Resumen, ejercicios y referencias

### Resumen
Construimos un transformer decoder-only completo desde el producto punto hasta la generación: self-attention escalado, causal masking, multi-head, RoPE, RMSNorm, residuales, FFN, y las variantes modernas (KV-cache, GQA, Flash Attention, MoE). El MiniGPT pasó la prueba del overfit, entrenó y generó texto técnico, y visualizamos su interior.

### Ejercicios
1. **(★)** Implementa el positional encoding *learned* (`nn.Embedding(max_seq_len, d_model)`) y compáralo con RoPE en el overfit de un batch.
2. **(★★)** Añade KV-cache real a `generate()` (guardando k,v por capa) y mide el speedup vs la versión naïve.
3. **(★★)** Sustituye la FFN por SwiGLU ajustando `d_ff` a `2/3·4·d_model` para igualar parámetros; ¿cambia la loss final?
4. **(★★★)** Integra el `MoEFFN` en el bloque y entrena; grafica la utilización de expertos con y sin loss de balanceo.
5. **(★★★)** Reemplaza `sdpa_scratch` por `F.scaled_dot_product_attention` en todo el modelo y mide memoria/velocidad en GPU.

### Referencias
- Vaswani et al. (2017). *Attention Is All You Need*. [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
- Su et al. (2021). *RoFormer: Enhanced Transformer with Rotary Position Embedding (RoPE)*. [arXiv:2104.09864](https://arxiv.org/abs/2104.09864)
- Ainslie et al. (2023). *GQA: Training Generalized Multi-Query Transformer Models*. [arXiv:2305.13245](https://arxiv.org/abs/2305.13245)
- Dao et al. (2022). *FlashAttention*. [arXiv:2205.14135](https://arxiv.org/abs/2205.14135)
- Shazeer (2020). *GLU Variants Improve Transformer (SwiGLU)*. [arXiv:2002.05202](https://arxiv.org/abs/2002.05202)
- Fedus et al. (2021). *Switch Transformers (MoE)*. [arXiv:2101.03961](https://arxiv.org/abs/2101.03961)
- Radford et al. (2019). *Language Models are Unsupervised Multitask Learners (GPT-2)*.
- Touvron et al. (2023). *Llama 2*. [arXiv:2307.09288](https://arxiv.org/abs/2307.09288)

> El código limpio y modular de todo esto está en [`src/llm_lab/model/`](../src/llm_lab/model/).